# Daily Challenge: Preprocess & Fine-Tune Transformer-Based Models

## 1. Understanding BERT and XLM-RoBERTa

**BERT (Bidirectional Encoder Representations from Transformers)**

BERT is an encoder-only Transformer model that reads text bidirectionally, meaning it considers both the words before and after a given token to build its representation. It was pretrained on English text using masked language modeling (predicting randomly masked words) and next-sentence prediction. Popular pretrained versions include `bert-base-uncased` (12 layers, lowercase, English), `bert-large-uncased` (24 layers, more capacity), and `bert-base-multilingual-cased` (supports 104 languages).

**XLM-RoBERTa**

XLM-RoBERTa is a multilingual extension of RoBERTa (itself an optimized version of BERT), pretrained on text from 100 languages using a single shared vocabulary and architecture. Unlike BERT, it does not use next-sentence prediction during pretraining, relying solely on masked language modeling but at a much larger scale and with more training data. Pretrained versions include `xlm-roberta-base` (12 layers) and `xlm-roberta-large` (24 layers), both useful for cross-lingual tasks like multilingual classification, translation-related tasks, or any application that must support many languages with a single model.

**How tokenization fits in**

Both models cannot process raw text directly — their tokenizers first split text into subword units (WordPiece for BERT, SentencePiece/BPE for XLM-RoBERTa) and map them to integer IDs from the model's vocabulary, which the model's embedding layer then converts into vectors.

In [ ]:
!pip install --quiet transformers torch scikit-learn pandas

In [ ]:
from transformers import BertTokenizer, XLMRobertaTokenizer

bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
xlmr_tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

print("BERT vocab size       :", bert_tokenizer.vocab_size)
print("XLM-RoBERTa vocab size:", xlmr_tokenizer.vocab_size)
print("\nBERT special tokens       :", bert_tokenizer.special_tokens_map)
print("XLM-RoBERTa special tokens:", xlmr_tokenizer.special_tokens_map)

## 2. Tokenizing Text

In [ ]:
single_sentence = "Transformer models have changed how we approach natural language processing."

# Single-sentence tokenization with BERT
bert_encoded = bert_tokenizer.encode_plus(
    single_sentence,
    add_special_tokens=True,
    return_tensors="pt"
)

print("BERT — Single sentence encoding")
print("input_ids:", bert_encoded["input_ids"])
print("attention_mask:", bert_encoded["attention_mask"])
print("Decoded:", bert_tokenizer.decode(bert_encoded["input_ids"][0]))

In [ ]:
# Single-sentence tokenization with XLM-RoBERTa
xlmr_encoded = xlmr_tokenizer.encode_plus(
    single_sentence,
    add_special_tokens=True,
    return_tensors="pt"
)

print("XLM-RoBERTa — Single sentence encoding")
print("input_ids:", xlmr_encoded["input_ids"])
print("attention_mask:", xlmr_encoded["attention_mask"])
print("Decoded:", xlmr_tokenizer.decode(xlmr_encoded["input_ids"][0]))

In [ ]:
# Two-sentence tokenization (e.g., for tasks like sentence-pair classification)
sentence_a = "The weather today is sunny and warm."
sentence_b = "It is a perfect day for a walk outside."

bert_pair_encoded = bert_tokenizer.encode_plus(
    sentence_a, sentence_b,
    add_special_tokens=True,
    return_tensors="pt"
)

print("BERT — Two-sentence encoding")
print("input_ids:", bert_pair_encoded["input_ids"])
print("token_type_ids:", bert_pair_encoded["token_type_ids"])
print("Decoded:", bert_tokenizer.decode(bert_pair_encoded["input_ids"][0]))

In [ ]:
xlmr_pair_encoded = xlmr_tokenizer.encode_plus(
    sentence_a, sentence_b,
    add_special_tokens=True,
    return_tensors="pt"
)

print("XLM-RoBERTa — Two-sentence encoding")
print("input_ids:", xlmr_pair_encoded["input_ids"])
print("Decoded:", xlmr_tokenizer.decode(xlmr_pair_encoded["input_ids"][0]))

**Observed token types**

- **`input_ids`**: The numerical IDs representing each token (including special tokens) that the model's embedding layer will look up.
- **`attention_mask`**: A binary mask (1 for real tokens, 0 for padding) telling the model which positions to actually attend to.
- **`token_type_ids`** (BERT only): Distinguishes which sentence each token belongs to in a two-sentence input (0 for the first sentence, 1 for the second). XLM-RoBERTa instead separates sentences using two `</s>` tokens rather than `token_type_ids`.
- **`labels`**: Not produced by the tokenizer itself — these are the target class IDs we supply separately during supervised fine-tuning, paired with each tokenized example.

## 3. Preparing Input Data for the Model

In [ ]:
max_length = 32

# BERT — with padding and truncation
bert_padded = bert_tokenizer.encode_plus(
    single_sentence,
    add_special_tokens=True,
    max_length=max_length,
    padding="max_length",
    truncation=True,
    return_attention_mask=True,
    return_tensors="pt"
)

print("BERT padded input_ids shape:", bert_padded["input_ids"].shape)
print("BERT attention_mask:", bert_padded["attention_mask"])

In [ ]:
# XLM-RoBERTa — with padding and truncation
xlmr_padded = xlmr_tokenizer.encode_plus(
    single_sentence,
    add_special_tokens=True,
    max_length=max_length,
    padding="max_length",
    truncation=True,
    return_attention_mask=True,
    return_tensors="pt"
)

print("XLM-RoBERTa padded input_ids shape:", xlmr_padded["input_ids"].shape)
print("XLM-RoBERTa attention_mask:", xlmr_padded["attention_mask"])

In [ ]:
print("BERT special tokens:")
print(bert_tokenizer.special_tokens_map)
print("\nXLM-RoBERTa special tokens:")
print(xlmr_tokenizer.special_tokens_map)

print("\nBERT vocab size       :", bert_tokenizer.vocab_size)
print("XLM-RoBERTa vocab size:", xlmr_tokenizer.vocab_size)

**Special tokens `<s>` and `</s>`**

XLM-RoBERTa (like RoBERTa) uses `<s>` to mark the beginning of a sequence and `</s>` to mark its end, analogous to BERT's `[CLS]` and `[SEP]` tokens. For sentence-pair inputs, XLM-RoBERTa separates the two sentences with `</s></s>` (a doubled separator) rather than using `token_type_ids` like BERT does. The `attention_mask` is essential in both models: padding tokens (added to reach `max_length`) carry no real information, and without the mask, the self-attention mechanism would otherwise let real tokens attend to meaningless padding positions, diluting the model's representations and degrading performance.

## 4. Loading and Exploring the Dataset

In [ ]:
import urllib.request, zipfile, os, glob
import pandas as pd

url = "https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%206/W6D1%20GenAi%20France/Basics%20of%20BERT%20and%20XLM-RoBERTa%20-%20PyTorch%20-%202.zip"
urllib.request.urlretrieve(url, "bert_xlmr_data.zip")

with zipfile.ZipFile("bert_xlmr_data.zip", "r") as z:
    z.extractall("bert_xlmr_data")

csv_files = glob.glob("bert_xlmr_data/**/*.csv", recursive=True)
print("CSV files found:", csv_files)

In [ ]:
# Identify train and test files by name
train_file = next((f for f in csv_files if "train" in os.path.basename(f).lower()), csv_files[0])
test_file = next((f for f in csv_files if "test" in os.path.basename(f).lower()), None)

train_df = pd.read_csv(train_file)
print(f"Loaded training file: {train_file}")
print(f"Shape: {train_df.shape}")

if test_file:
    test_df = pd.read_csv(test_file)
    print(f"\nLoaded test file: {test_file}")
    print(f"Shape: {test_df.shape}")
else:
    test_df = None
    print("\nNo separate test file found.")

In [ ]:
train_df.head()

In [ ]:
print("Columns:", train_df.columns.tolist())
print("\nData types:")
print(train_df.dtypes)
print("\nMissing values:")
print(train_df.isnull().sum())

In [ ]:
# Identify the text and label columns
text_col = next((c for c in train_df.columns if c.lower() in ["text", "sentence", "review", "comment"]),
                 train_df.select_dtypes(include="object").columns[0])
label_col = next((c for c in train_df.columns if c.lower() in ["label", "target", "class", "sentiment"]),
                  train_df.columns[-1])

print(f"Text column : '{text_col}'")
print(f"Label column: '{label_col}'")
print(f"\nLabel distribution:")
print(train_df[label_col].value_counts())

## 5. Creating Cross-Validation Folds

In [ ]:
from sklearn.model_selection import StratifiedKFold

N_FOLDS = 5

kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

X = train_df[text_col].values
y = train_df[label_col].values

fold_splits = []

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    fold_splits.append({
        "fold": fold_idx,
        "train_idx": train_idx,
        "val_idx": val_idx
    })
    print(f"Fold {fold_idx}: train={len(train_idx)} samples, val={len(val_idx)} samples")

In [ ]:
# Verify that label distribution is preserved across folds
for split in fold_splits:
    fold = split["fold"]
    train_labels = y[split["train_idx"]]
    val_labels = y[split["val_idx"]]

    train_dist = pd.Series(train_labels).value_counts(normalize=True).sort_index()
    val_dist = pd.Series(val_labels).value_counts(normalize=True).sort_index()

    print(f"\nFold {fold} — Label distribution (proportion)")
    print(f"  Train: {train_dist.to_dict()}")
    print(f"  Val  : {val_dist.to_dict()}")

In [ ]:
# Build actual train/val DataFrame splits for each fold, ready for tokenization and training
fold_dataframes = []

for split in fold_splits:
    fold_train_df = train_df.iloc[split["train_idx"]].reset_index(drop=True)
    fold_val_df = train_df.iloc[split["val_idx"]].reset_index(drop=True)
    fold_dataframes.append({
        "fold": split["fold"],
        "train_df": fold_train_df,
        "val_df": fold_val_df
    })

print(f"Created {len(fold_dataframes)} fold DataFrame splits, ready for tokenization and fine-tuning.")
print(f"\nExample — Fold 0 train shape: {fold_dataframes[0]['train_df'].shape}")
print(f"Example — Fold 0 val shape  : {fold_dataframes[0]['val_df'].shape}")

## 6. Tokenizing the Dataset for Fine-Tuning

In [ ]:
def tokenize_dataframe(df, tokenizer, text_column, max_length=64):
    """
    Tokenizes all texts in a DataFrame column, returning padded and
    truncated input_ids and attention_mask tensors ready for training.
    """
    return tokenizer(
        df[text_column].tolist(),
        add_special_tokens=True,
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_tensors="pt"
    )


# Example: tokenize fold 0 with BERT
fold0_train_encoded = tokenize_dataframe(fold_dataframes[0]["train_df"], bert_tokenizer, text_col)
fold0_val_encoded = tokenize_dataframe(fold_dataframes[0]["val_df"], bert_tokenizer, text_col)

print("Fold 0 — Tokenized train input_ids shape:", fold0_train_encoded["input_ids"].shape)
print("Fold 0 — Tokenized val input_ids shape  :", fold0_val_encoded["input_ids"].shape)

## 7. Fine-Tuning a Transformer Model (BERT)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertForSequenceClassification

num_labels = train_df[label_col].nunique()
print(f"Number of classes: {num_labels}")


class TextClassificationDataset(Dataset):
    """Wraps tokenized inputs and labels into a PyTorch Dataset."""
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
def run_fold_training(fold_data, tokenizer, model_name, label_col, text_col,
                       num_labels, max_length=64, epochs=2, batch_size=16):
    """
    Fine-tunes a transformer model on a single cross-validation fold and
    returns the validation accuracy for that fold.
    """
    train_df_fold = fold_data["train_df"]
    val_df_fold = fold_data["val_df"]

    train_encodings = tokenize_dataframe(train_df_fold, tokenizer, text_col, max_length)
    val_encodings = tokenize_dataframe(val_df_fold, tokenizer, text_col, max_length)

    train_dataset = TextClassificationDataset(train_encodings, train_df_fold[label_col].values)
    val_dataset = TextClassificationDataset(val_encodings, val_df_fold[label_col].values)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    model = BertForSequenceClassification.from_pretrained(model_name, num_labels=num_labels).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"  Epoch {epoch + 1}/{epochs} — Avg train loss: {total_loss / len(train_loader):.4f}")

    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            preds = torch.argmax(outputs.logits, dim=-1)
            correct += (preds == batch["labels"]).sum().item()
            total += batch["labels"].size(0)

    val_accuracy = correct / total
    return val_accuracy


print("run_fold_training function defined.")

In [ ]:
# Run cross-validation across all folds (using a small number of epochs for demonstration)
fold_accuracies = []

for fold_data in fold_dataframes:
    print(f"\n{'='*50}\nTraining Fold {fold_data['fold']}\n{'='*50}")
    acc = run_fold_training(
        fold_data,
        bert_tokenizer,
        "bert-base-uncased",
        label_col,
        text_col,
        num_labels,
        max_length=64,
        epochs=2,
        batch_size=16
    )
    fold_accuracies.append(acc)
    print(f"Fold {fold_data['fold']} — Validation Accuracy: {acc:.4f}")

In [ ]:
import numpy as np

print("Cross-Validation Results (BERT)")
print("-" * 40)
for i, acc in enumerate(fold_accuracies):
    print(f"Fold {i}: {acc:.4f}")

print(f"\nMean accuracy: {np.mean(fold_accuracies):.4f}")
print(f"Std accuracy : {np.std(fold_accuracies):.4f}")

**Why cross-validation matters here**

Using `StratifiedKFold` ensures that each of the 5 folds maintains roughly the same class distribution as the full dataset, which is important for getting a reliable estimate of model performance, especially if the dataset has any class imbalance. Averaging the validation accuracy across all 5 folds gives a more robust estimate of how well the fine-tuned model generalizes than a single train/validation split would, and the standard deviation across folds indicates how sensitive the model's performance is to which specific examples end up in the training versus validation set.